# Phase 2A v2 reread sample pipeline

This notebook follows the execution order of `phase2A.ipynb`, but writes a simpler output set.

Main changes:

- Start with sample-population descriptive statistics.
- Write normal CSV headers only.
- Produce one row-level master file for all 1,155 presented image rows.
- Use only two row-level status fields: `read_type` and `selection_role`.
- Do not export intermediate implementation labels.


## 3.6 Input integrity and excluded-bundle checks

Normalize the source CSV and verify the Phase 1 analysis-set shape. The excluded 2026 bundles are checked by absence rules before any selection step.


In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
try:
    from scipy.stats import hypergeom, ks_2samp, mannwhitneyu
except ImportError:
    from types import SimpleNamespace
    class hypergeom:
        @staticmethod
        def pmf(x, M, n, N):
            if x != 0:
                raise NotImplementedError("fallback only supports pmf(0, M, n, N)")
            k = np.asarray(n)
            out = np.zeros(k.shape, dtype=float)
            denom = math.comb(int(M), int(N)) if 0 <= int(N) <= int(M) else 0
            for idx, kk in np.ndenumerate(k):
                kk = int(kk)
                if denom and 0 <= N <= M - kk:
                    out[idx] = math.comb(int(M - kk), int(N)) / denom
                else:
                    out[idx] = 0.0
            return out
    def ks_2samp(a, b):
        return SimpleNamespace(pvalue=np.nan)
    def mannwhitneyu(a, b):
        return SimpleNamespace(pvalue=np.nan)

SOURCE_CSV = Path("CXR/unified_labels.csv")
ROIS = ["RT", "LT", "RB", "LB"]
VALID_GRADES = {0, 1, 2, 3, 4}
MS_LEN = 7                       # 3.4 — 원 시퀀스 최소값. 연도 간 제시 범위 대칭을 위해 고정
pd.set_option("display.max_rows", 250, "display.width", 240)

raw = pd.read_csv(SOURCE_CSV, encoding="utf-8-sig")
raw.columns = [str(c).strip() for c in raw.columns]
df = raw.rename(columns={"patient_id": "pid"}).copy()
df["pid"] = df.pid.astype(str).str.strip()
df["uid"] = df.uid.astype(str).str.strip()
df["seq"] = pd.to_numeric(df.seq, errors="coerce")
df["year"] = df.uid.str[:2].map({"24": 2024, "26": 2026})
for roi in ROIS:
    df[f"orig_{roi}"] = pd.to_numeric(df[roi], errors="coerce")

integrity = {
    "rows": len(df), "patients": df.pid.nunique(),
    "missing_seq": int(df.seq.isna().sum()),
    "invalid_year": int((~df.year.isin([2024, 2026])).sum()),
    "duplicate_uid": int(df.uid.duplicated().sum()),
    "duplicate_pid_seq": int(df.duplicated(["pid", "seq"]).sum()),
    "pid_in_both_years": int((df.groupby("pid").year.nunique() > 1).sum()),
}
for roi in ROIS:
    integrity[f"missing_{roi}"] = int(df[f"orig_{roi}"].isna().sum())
    integrity[f"invalid_{roi}"] = int((~df[f"orig_{roi}"].isin(VALID_GRADES)).sum())
display(pd.Series(integrity, name="value").to_frame())
if any(v for k, v in integrity.items() if k not in {"rows", "patients"}):
    raise ValueError("입력 검증 실패")

df["seq"] = df.seq.astype(int)
df["year"] = df.year.astype(int)
for roi in ROIS:
    df[f"orig_{roi}"] = df[f"orig_{roi}"].astype(int)
df = df.sort_values(["pid", "seq"]).reset_index(drop=True)
df["seq_position"] = df.groupby("pid").cumcount() + 1
df["pid_n_images"] = df.groupby("pid")["seq"].transform("size")
PID_N = df.groupby("pid").pid_n_images.first().to_dict()
POS_UID = df.set_index(["pid", "seq_position"]).uid.to_dict()
IMG = df.set_index("uid")

cohort = df.groupby("year").agg(images=("uid", "size"), patients=("pid", "nunique"))
cohort["min_images_per_patient"] = df.groupby("year").pid_n_images.min()
display(cohort)

EXPECTED = {2024: (718, 68), 2026: (587, 46)}
for y, (ei, ep) in EXPECTED.items():
    got = (int(cohort.loc[y, "images"]), int(cohort.loc[y, "patients"]))
    assert got == (ei, ep), f"{y}: {got} != Phase 1 분석셋 {(ei, ep)}"

# 3.6 — 2026 누락 케이스 세 묶음의 부재를 각각 확인
missing_case_absence = pd.DataFrame([
    {"bundle": "미라벨 10명/126장",
     "check": "ROI 등급 결측 0 (미라벨 혼입 시 결측 행 발생)",
     "observed": sum(integrity[f"missing_{r}"] for r in ROIS), "criterion": 0},
    {"bundle": "재측정 8명/95장",
     "check": "uid 중복 0 그리고 (pid,seq) 중복 0 (재측정 혼입 시 같은 환자·순번 중복 발생)",
     "observed": integrity["duplicate_uid"] + integrity["duplicate_pid_seq"], "criterion": 0},
    {"bundle": "부분겹침 9명/96장",
     "check": "2026 환자 수 46 (혼입 시 55명 이상)",
     "observed": int(cohort.loc[2026, "patients"]), "criterion": 46},
])
missing_case_absence["verdict"] = np.where(
    missing_case_absence.observed == missing_case_absence.criterion, "부재 확인", "FAIL")
display(missing_case_absence)
if (missing_case_absence.verdict == "FAIL").any():
    raise ValueError("2026 누락 케이스 혼입 의심 — 입력 정규화 단계에서 중단 (3.6)")

min_n = int(df.pid_n_images.min())
print(f"환자별 최소 영상 수 = {min_n} (계획서 3.2: 두 코호트 모두 7) — "
      f"길이 {MS_LEN} mini-sequence 구성 가능")
assert min_n >= MS_LEN


## 1. Sample-population descriptive statistics

Recount the source population before any selection step: cohort size, patient image counts, ROI grade distribution, RB eligibility, and patient-level overlap.


In [ ]:
pop_rb = (df.groupby(["year", "orig_RB"])
          .agg(images=("uid", "size"), patients=("pid", "nunique")).reset_index())
print("RB 기존 등급별 모집단 분모")
display(pop_rb.pivot(index="orig_RB", columns="year", values="images"))

roi_long_pop = df.melt(id_vars=["year", "pid", "uid"],
                       value_vars=[f"orig_{r}" for r in ROIS],
                       var_name="ROI", value_name="orig_grade")
roi_long_pop["ROI"] = roi_long_pop.ROI.str.replace("orig_", "", regex=False)
print("ROI별 기존 등급별 모집단 분모")
display(roi_long_pop.pivot_table(index=["ROI", "year"], columns="orig_grade",
                                 values="uid", aggfunc="nunique", fill_value=0))

pe = df.groupby(["year", "pid"]).agg(
    n_images=("uid", "size"),
    target=("orig_RB", lambda s: bool(s.isin([3, 4]).any())),
    lower=("orig_RB", lambda s: bool((s == 2).any())),
    far=("orig_RB", lambda s: bool(s.isin([0, 1]).any())),
).reset_index()
pe["frame_ok"] = pe.n_images >= MS_LEN

group_eligibility = pe.groupby("year").agg(
    patients=("pid", "size"),
    target_eligible=("target", "sum"),
    lower_eligible=("lower", "sum"),
    far_eligible=("far", "sum"),
    frame_excluded=("frame_ok", lambda s: int((~s).sum())),
).reset_index()
print("대상군별 적격 환자 수")
display(group_eligibility)

patient_overlap = (pe[pe.year.eq(2024)]
                   .groupby(["target", "lower", "far"]).pid.count()
                   .reset_index(name="patients"))
print("2024 환자의 대상군 중첩 구조")
display(patient_overlap)

frame_exclusions = pe[~pe.frame_ok][["year", "pid", "n_images"]].copy()
frame_exclusions["reason"] = f"images < {MS_LEN}"
print(f"표집틀 제한(3.2)으로 대조군 후보에서 제외되는 환자: {len(frame_exclusions)}명")

census_ref = (df[df.orig_RB.isin([3, 4])]
              .groupby(["year", "orig_RB"])
              .agg(images=("uid", "size"), patients=("pid", "nunique")))
display(census_ref)
REF = {(2024, 3): 135, (2024, 4): 68, (2026, 3): 122, (2026, 4): 55}
for (y, g), exp in REF.items():
    got = int(census_ref.loc[(y, g), "images"])
    assert got == exp, f"{y} RB{g}: {got}장 != Phase 1 참조 {exp}장"
print("Phase 1 기준 참조 개수 일치: 표적군 203장 · 원판독 조건 대조군 177장 (3.2)")

YEAR_CONTROL_DESIGN = "census"   # 3.2 — 분모 177이 표적군 203과 동등 규모이므로 전수 포함
print(f"원판독 조건 대조군 결정: {YEAR_CONTROL_DESIGN} (전수 포함) — "
      "표집가중 불필요, 두 비교 조건의 제시 범위 대칭 유지 (3.4)")


## 3.4 Census mini-sequences for RB grade 3-4

Construct the minimum number of length-7 mini-sequences needed to include all 2024 and 2026 RB grade 3-4 images.


In [ ]:
RANDOM_SEED = 42          # 재현성 기록용 메타데이터 (3.4)
rng_census = np.random.default_rng(RANDOM_SEED)
rng_sample = np.random.default_rng(RANDOM_SEED + 1)
rng_dup = np.random.default_rng(RANDOM_SEED + 2)
rng_order = np.random.default_rng(RANDOM_SEED + 3)

def census_windows(n, targets, rng):
    """길이 MS_LEN 창의 최소 개수로 targets(1-index 위치) 전부를 덮는다.
    greedy 가 최소 개수를 정하고, 각 창의 시작 위치 잔여 자유도는 rng 로 확정한다(3.4)."""
    remaining = sorted(targets)
    assigned = []
    while remaining:
        t = remaining[0]
        s = min(t, n - MS_LEN + 1)
        assigned.append([x for x in remaining if x <= s + MS_LEN - 1])
        remaining = [x for x in remaining if x > s + MS_LEN - 1]
    starts = []
    for cov in assigned:
        lo = max(1, max(cov) - MS_LEN + 1)
        hi = min(min(cov), n - MS_LEN + 1)
        pool = [s for s in range(lo, hi + 1) if s not in starts]
        if not pool:
            pool = list(range(lo, hi + 1))
        starts.append(int(rng.choice(pool)))
    return starts

census_list = []
for (year, pid), rows in df.groupby(["year", "pid"], sort=True):
    tpos = rows.loc[rows.orig_RB.isin([3, 4]), "seq_position"].astype(int).tolist()
    if not tpos:
        continue
    n = PID_N[pid]
    starts = census_windows(n, tpos, rng_census)
    covered = set()
    for s in starts:
        covered.update(range(s, s + MS_LEN))
    assert set(tpos) <= covered, f"{pid}: 전수 포함 실패"
    for s in starts:
        census_list.append({"year": year, "pid": pid, "start": s})
cms = pd.DataFrame(census_list)

for year in (2024, 2026):
    sub = cms[cms.year.eq(year)]
    cov_uids = set()
    for r in sub.itertuples():
        cov_uids.update(POS_UID[(r.pid, p)] for p in range(r.start, r.start + MS_LEN))
    t_uids = set(df[(df.year.eq(year)) & df.orig_RB.isin([3, 4])].uid)
    assert t_uids <= cov_uids, f"{year}: 전수 포함 검증 실패"
    n_pairs = sub.groupby("pid").size()
    print(f"{year}: 전수 포함 mini-sequence {len(sub)}개 · 환자 {sub.pid.nunique()}명 "
          f"(환자당 최대 {int(n_pairs.max())}개) · RB 3·4 전수 {len(t_uids)}장 포함 확인")


## 3.4 Control mini-sequence candidate population

Build the 2024 length-7 mini-sequence candidate population used by the proportional balanced control sampling step.


In [ ]:
frame_pids = pe[(pe.year == 2024) & pe.frame_ok].pid.tolist()
G_BY_PID = {pid: df[df.pid.eq(pid)].sort_values("seq_position").orig_RB.to_numpy()
            for pid in frame_pids}

cand_rows = []
for pid in frame_pids:
    g = G_BY_PID[pid]
    n = len(g)
    for s in range(1, n - MS_LEN + 2):
        w = g[s - 1:s - 1 + MS_LEN]
        cand_rows.append({"pid": pid, "start": s,
                          "has_g2": bool((w == 2).any()),
                          "has_g01": bool(np.isin(w, [0, 1]).any())})
cand = pd.DataFrame(cand_rows)
N_ALL = len(cand)
print(f"2024 길이 {MS_LEN} 창 후보 모집단 {N_ALL}개 — "
      f"RB 등급2 포함 {int(cand.has_g2.sum())}개 · RB 0·1 포함 {int(cand.has_g01.sum())}개")

img24 = df[df.year.eq(2024)][["uid", "pid", "seq_position", "pid_n_images", "orig_RB"]].copy()

k_all = []
starts_by_pid = {}
for r in cand.itertuples():
    starts_by_pid.setdefault(r.pid, set()).add(r.start)
for r in img24.itertuples():
    lo = max(1, r.seq_position - MS_LEN + 1)
    hi = min(r.seq_position, r.pid_n_images - MS_LEN + 1)
    k_all.append(sum(1 for s in range(lo, hi + 1) if s in starts_by_pid.get(r.pid, ())))
img24["k_all"] = np.array(k_all)

census_uids24 = set()
for r in cms[cms.year.eq(2024)].itertuples():
    census_uids24.update(POS_UID[(r.pid, p)] for p in range(r.start, r.start + MS_LEN))
img24["in_census"] = img24.uid.isin(census_uids24)
print(f"전수 포함 mini-sequence 내부(포함확률 1)의 2024 영상: {int(img24.in_census.sum())}장")
display(img24.groupby("orig_RB").agg(images=("uid", "size"),
                                     in_census=("in_census", "sum"),
                                     mean_k_all=("k_all", "mean")))


## 6.3 Control sampling: population-proportional balanced SRS

Select control mini-sequences so the selected 2024 ROI x original-grade distribution approximates the 2024 population distribution.


In [ ]:
PRESENTATION_BUDGET = 1050       # 사전 지정 — 원본 제시 영상 예산, 단위: 장 (조정 가능)
N_REJECTIVE = 1000               # 균형 SRS 반복 수 (기록용)

census_rows = MS_LEN * len(cms)
m_ctrl = max(0, (PRESENTATION_BUDGET - census_rows) // MS_LEN)
print(f"원본 제시 영상 예산 {PRESENTATION_BUDGET}장 − 전수 {census_rows}장 → "
      f"표집 mini-sequence m = {m_ctrl} (후보 {N_ALL}개)")

# 목표 분포: 2024 모집단 ROI×등급 비율 (20-셀)
def onehot(uid):
    v = np.zeros(len(ROIS) * 5)
    row = IMG.loc[uid]
    for ri, r in enumerate(ROIS):
        v[ri * 5 + int(row[f"orig_{r}"])] += 1
    return v

uids24 = df[df.year.eq(2024)].uid.tolist()
U24 = {u: onehot(u) for u in uids24}
pop_vec = sum(U24.values())
pop_prop = pop_vec / len(uids24)

census_vec24 = sum(U24[u] for u in census_uids24)
win_uids = {(r.pid, r.start): [POS_UID[(r.pid, p)] for p in range(r.start, r.start + MS_LEN)]
            for r in cand.itertuples()}
keys_all = list(win_uids)

def selected_L1(sample_keys):
    new = set()
    for k in sample_keys:
        new.update(win_uids[k])
    new -= census_uids24
    vec = census_vec24 + (sum(U24[u] for u in new) if new else 0)
    n = len(census_uids24) + len(new)
    return float(np.abs(vec / n - pop_prop).sum()), new

best_L1, best_sample, l1s = None, None, []
for _ in range(N_REJECTIVE):
    idx = rng_sample.choice(N_ALL, size=m_ctrl, replace=False)
    sample = [keys_all[i] for i in idx]
    l1, _ = selected_L1(sample)
    l1s.append(l1)
    if best_L1 is None or l1 < best_L1:
        best_L1, best_sample = l1, sample
ctrl_sel = pd.DataFrame([{"pid": p, "start": s} for p, s in best_sample])
_, ctrl_new_uids = selected_L1(best_sample)

census_only_L1 = float(np.abs(census_vec24 / len(census_uids24) - pop_prop).sum())
print(f"균형 SRS {N_REJECTIVE}회 — L1 편차: 전수만 {census_only_L1:.3f} · "
      f"SRS 평균 {np.mean(l1s):.3f} · 채택 표본 {best_L1:.3f}")
print(f"표집 창 {len(ctrl_sel)}개 → 신규 고유 영상 {len(ctrl_new_uids)}장")

sampling_decision = pd.DataFrame([{
    "group": "proportional_control (하위 경계·원거리 분모 자격 공급)",
    "frame_N": N_ALL, "m_chosen": m_ctrl,
    "rule": ("모집단 비례 균형 표집 — 전체 창 SRS 를 균형(rejective) 채택: "
             f"{N_REJECTIVE}회 SRS 중 선정 세트 ROIx등급 분포와 2024 모집단 분포의 "
             "L1 편차 최소 표본. 분모 수치 목표 없음 — 분모는 결과로 보고. "
             "계획서 v5 6.3 의 모집단 비례 균형 표집 규칙"),
    "L1_census_only": round(census_only_L1, 4),
    "L1_srs_mean": round(float(np.mean(l1s)), 4),
    "L1_accepted": round(best_L1, 4),
    "presentations_added": m_ctrl * MS_LEN,
    "budget": PRESENTATION_BUDGET,
}])
display(sampling_decision.T)


## 4.1 Selected original set and inclusion probabilities

Assemble selected original mini-sequences, derive image-level entry paths, joint inclusion probabilities, weight strata, and Kish effective sample sizes.


In [ ]:
sel_windows = {}
def add_window(year, pid, start, path):
    sel_windows.setdefault((year, pid, start), set()).add(path)

for r in cms.itertuples():
    add_window(r.year, r.pid, r.start, "census")
for r in ctrl_sel.itertuples():
    add_window(2024, r.pid, r.start, "ctrl")

qual = cand.set_index(["pid", "start"])[["has_g2", "has_g01"]]
P_CTRL = m_ctrl / N_ALL if N_ALL else 0.0

ms_info_rows, occ_list = [], []
for i, ((year, pid, start), paths) in enumerate(sorted(sel_windows.items()), 1):
    ms_key = f"SRC{i:04d}"
    pi_ms = 1.0 if "census" in paths else P_CTRL
    ms_info_rows.append({"ms_key": ms_key, "year": year, "pid": pid, "start": start,
                         "paths": "|".join(sorted(paths)), "pi_ms": pi_ms,
                         "ms_len": MS_LEN, "exposure": MS_LEN / PID_N[pid]})
    ctrl_codes = set()
    if "ctrl" in paths:
        q = qual.loc[(pid, start)]
        if bool(q.has_g2):
            ctrl_codes.add("lower_cutpoint_sampled")
        if bool(q.has_g01):
            ctrl_codes.add("far_control_sampled")
    for p in range(start, start + MS_LEN):
        uid = POS_UID[(pid, p)]
        rb = int(IMG.loc[uid, "orig_RB"])
        codes = set(ctrl_codes)
        if "census" in paths:
            if year == 2024:
                codes.add("target_census" if rb in (3, 4) else "target_ms_interior")
            else:
                codes.add("year_control_census")
        occ_list.append({"ms_key": ms_key, "year": year, "pid": pid, "uid": uid,
                         "seq2": p - start + 1, "entry_codes": codes})
ms_info = pd.DataFrame(ms_info_rows)
occ = pd.DataFrame(occ_list)
print(f"원본 mini-sequence {len(ms_info)}개 · 제시 {len(occ)}장 · 고유 영상 {occ.uid.nunique()}장")
display(ms_info.groupby(["year", "paths"]).agg(n_ms=("ms_key", "size"),
                                               patients=("pid", "nunique")))

# ── 영상 수준: 선정 경로 합집합과 결합 포함확률 (4.1) ────────────────────
def pi_srs(N, k, m):
    """후보 N개 중 m개 SRS 에서, k개 창에 덮이는 영상의 포함확률 (균형 채택 근사)."""
    k = np.atleast_1d(np.asarray(k))
    if m <= 0:
        return np.zeros(k.shape)
    if m >= N:
        return (k > 0).astype(float)
    return 1.0 - hypergeom.pmf(0, N, k, m)

entry_by_uid = (occ.groupby("uid").entry_codes
                .apply(lambda s: "|".join(sorted(set().union(*s)))))
sel_uids = occ.uid.unique()
img_sel = df[df.uid.isin(sel_uids)][
    ["uid", "pid", "seq", "seq_position", "year"] + [f"orig_{r}" for r in ROIS]].copy()
img_sel["entry_path"] = img_sel.uid.map(entry_by_uid)

k_map = img24.set_index("uid")[["k_all", "in_census"]]
img_sel = img_sel.merge(k_map, left_on="uid", right_index=True, how="left")
img_sel["k_all"] = img_sel.k_all.fillna(0).astype(int)
img_sel["prob1_path"] = img_sel.in_census.fillna(True).astype(bool)   # 2026 전수 포함

pi_ctrl_i = pi_srs(N_ALL, img_sel.k_all.to_numpy(), m_ctrl)
img_sel["pi_uid"] = np.where(img_sel.prob1_path, 1.0, pi_ctrl_i)
img_sel["weight_stratum"] = "pi=" + img_sel.pi_uid.round(6).astype(str)

print("가중 층 (결합 포함확률 값으로 정의, 4.1) — 포함확률은 SRS 식이며 균형 채택 근사")
strata = (img_sel.groupby("weight_stratum")
          .agg(images=("uid", "size"), patients=("pid", "nunique"),
               weight_sum=("pi_uid", lambda s: float((1 / s).sum())))
          .sort_index(ascending=False))
display(strata)

def kish_neff(pi):
    w = 1.0 / np.asarray(pi)
    return float(w.sum() ** 2 / (w ** 2).sum()) if len(w) else 0.0

kish_rows = []
for g in (0, 1, 2):
    sub = img_sel[(img_sel.year.eq(2024)) & (img_sel.orig_RB.eq(g))]
    kish_rows.append({"denominator": f"2024 RB grade {g}", "images": len(sub),
                      "patients": sub.pid.nunique(),
                      "weight_sum": round(float((1 / sub.pi_uid).sum()), 1),
                      "kish_neff": round(kish_neff(sub.pi_uid), 1)})
kish_table = pd.DataFrame(kish_rows)
print("표집 유래 분모의 실현 규모와 Kish 유효표본 수 (4.1) — 분모는 표집의 결과")
display(kish_table)

# ── 선정 세트 vs 모집단 ROI×등급 분포 대조 (표집 원칙 검증) ────────────────
sel24 = img_sel[img_sel.year.eq(2024)]
sel_vec = sum(onehot(u) for u in sel24.uid)
sel_prop = sel_vec / len(sel24)
distribution_match = pd.DataFrame(
    [{"ROI": r, "orig_grade": g,
      "population_prop": round(float(pop_prop[ri * 5 + g]), 4),
      "selected_prop": round(float(sel_prop[ri * 5 + g]), 4),
      "selected_n": int(sel_vec[ri * 5 + g])}
     for ri, r in enumerate(ROIS) for g in range(5)])
distribution_match["deviation"] = (distribution_match.selected_prop
                                   - distribution_match.population_prop).round(4)
print("2024 선정 세트 vs 모집단 ROI×등급 분포 (양수 편차 = 과대표집; RB·상위 등급의 "
      "과대는 전수 포함의 설계적 결과)")
display(distribution_match.pivot(index="orig_grade", columns="ROI", values="deviation"))
print(f"L1 편차 합 {float(np.abs(sel_prop - pop_prop).sum()):.3f} "
      f"(전수만일 때 {census_only_L1:.3f})")


## 4.1 and 4.3 Denominator roles derived from original grade

Derive denominator eligibility from old grade and boundary definitions, independent of selection path.


In [ ]:
DERIVE = {4: ["34_down"], 3: ["34_up", "23_down"], 2: ["23_up"],
          1: ["01_down"], 0: ["01_up"]}

obs = img_sel.melt(id_vars=["uid", "pid", "year", "weight_stratum", "pi_uid"],
                   value_vars=[f"orig_{r}" for r in ROIS],
                   var_name="ROI", value_name="orig_grade")
obs["ROI"] = obs.ROI.str.replace("orig_", "", regex=False)
obs["terms"] = obs.orig_grade.map(lambda g: DERIVE.get(int(g), []))

for roi in ROIS:
    img_sel[f"denom_role_{roi}"] = img_sel[f"orig_{roi}"].map(
        lambda g: "|".join(DERIVE.get(int(g), [])))

den = (obs.explode("terms").dropna(subset=["terms"])
       .groupby(["ROI", "year", "terms"])
       .agg(images=("uid", "nunique"), patients=("pid", "nunique")).reset_index())
print("도출된 분모 — 이미지 수")
display(den.pivot_table(index=["ROI", "year"], columns="terms",
                        values="images", fill_value=0))

def term_n(roi, year, term):
    q = den[(den.ROI == roi) & (den.year == year) & (den.terms == term)]
    return int(q.images.iloc[0]) if len(q) else 0

param_rows = [
    {"parameter": "Δ_2024,RB,34 (공동 주)", "up": term_n("RB", 2024, "34_up"),
     "down": term_n("RB", 2024, "34_down")},
    {"parameter": "Δ_2026,RB,34 (Δ_year 성분)", "up": term_n("RB", 2026, "34_up"),
     "down": term_n("RB", 2026, "34_down")},
    {"parameter": "Δ_2024,RB,23 (Δ_cutpoint 성분)", "up": term_n("RB", 2024, "23_up"),
     "down": term_n("RB", 2024, "23_down")},
    {"parameter": "Δ_far = Δ_2024,RB,01", "up": term_n("RB", 2024, "01_up"),
     "down": term_n("RB", 2024, "01_down")},
] + [
    {"parameter": f"Δ_2024,{roi},34 (ROI contrast, RB 조건부)",
     "up": term_n(roi, 2024, "34_up"), "down": term_n(roi, 2024, "34_down")}
    for roi in ["RT", "LT", "LB"]
]
param_denominators = pd.DataFrame(param_rows)
param_denominators["min_denominator"] = param_denominators[["up", "down"]].min(axis=1)
print("4.3·4.5 모수의 분모 — 분포 정합 표집의 결과; "
      "RT·LT·LB 분모는 RB 기반 선정에 조건부 (4.5.1)")
display(param_denominators)

# ── 산출물 검증 ────────────────────────────────────────────────────────
check_rows, failed = [], []
for roi in ROIS:
    for y in (2024, 2026):
        sub = obs[(obs.ROI == roi) & (obs.year == y)]
        rec = {"ROI": roi, "year": y}
        ok = True
        for g, terms in DERIVE.items():
            n_g = int((sub.orig_grade == g).sum())
            rec[f"grade{g}"] = n_g
            for t in terms:
                if term_n(roi, y, t) != n_g:
                    ok = False
        rec["dual_boundary"] = int(sub.terms.map(len).eq(2).sum())
        if rec["dual_boundary"] == 0:
            ok = False                                   # ④ 동시 기여 0건이면 실패
        rec["check"] = "OK" if ok else "FAIL"
        check_rows.append(rec)
        if not ok:
            failed.append((roi, y))
denominator_check = pd.DataFrame(check_rows)
display(denominator_check.set_index(["ROI", "year"]))
if failed:
    raise ValueError(f"산출물 검증 실패 (분모 소실): {failed}")

orphan = obs[obs.terms.map(len) == 0]
print(f"① 분모 밖 ROI 관측: {len(orphan)}건 — 모든 등급이 최소 한 분모에 속한다")

g2_rb = img_sel[(img_sel.year.eq(2024)) & (img_sel.orig_RB.eq(2))]
strata_sum = int(g2_rb.groupby("weight_stratum").size().sum())
assert strata_sum == len(g2_rb), "② 등급 2 가중 층 합 불일치"
print(f"② 2024 RB 등급 2 관측 {len(g2_rb)}장이 {g2_rb.weight_stratum.nunique()}개 "
      f"가중 층에 모두 속함 (층 합 {strata_sum})")

grade3_dual = pd.DataFrame([
    {"ROI": roi, "year": y, "grade3_obs": int(((obs.ROI == roi) & (obs.year == y)
                                               & (obs.orig_grade == 3)).sum()),
     "in_34_up": term_n(roi, y, "34_up"), "in_23_down": term_n(roi, y, "23_down")}
    for roi in ROIS for y in (2024, 2026)])
assert (grade3_dual.grade3_obs == grade3_dual.in_34_up).all()
assert (grade3_dual.grade3_obs == grade3_dual.in_23_down).all()
print("③ 등급 3 관측의 양쪽 분모 동시 소속 확인")
display(grade3_dual)

# ── 선정 세트 ROI×기존 등급 분모표 ────────────────────────────────────
selected_roi_grade = (obs.pivot_table(index=["ROI", "year"], columns="orig_grade",
                                      values="uid", aggfunc="nunique", fill_value=0)
                      .reset_index())
print("선정 세트 ROI×기존 등급 분모표 (고유 영상)")
display(selected_roi_grade.set_index(["ROI", "year"]))


## 3.5 and 6.4 Hidden duplicate size and allocation

Count overlap second presentations first. Then calculate the hidden-duplicate mini-sequence population and select mini-sequences from that population to improve ROI x original-grade balance. Record selected mini-sequence count and target-image count separately.


In [ ]:
# Hidden duplicate: overlap first, then select length-7 mini-sequences from the same population.
n_occ_per_uid = occ.groupby("uid").size()
overlap_uids = n_occ_per_uid[n_occ_per_uid > 1]

U_ALL = {u: (U24[u] if u in U24 else onehot(u)) for u in sel_uids}
target_vec = sum(U_ALL[u] for u in sel_uids) / len(sel_uids)

dup_vec = np.zeros(len(ROIS) * 5)
dup_n = 0
for uid, cnt in overlap_uids.items():
    dup_vec += U_ALL[uid] * (cnt - 1)
    dup_n += cnt - 1
overlap_n = dup_n
L1_before = float(np.abs(dup_vec / dup_n - target_vec).sum()) if dup_n else float("inf")
print(f"overlap second presentations: {overlap_n}; L1 before supplement {L1_before:.3f}")

original_patients = img_sel[["year", "pid"]].drop_duplicates().sort_values(["year", "pid"])
original_window_keys = set(sel_windows)
overlap_target_uids = set(overlap_uids.index)
eligible_target_uids = set(sel_uids) - overlap_target_uids

hd_pop_rows = []
for year, pid in original_patients.itertuples(index=False):
    n = PID_N[pid]
    for start in range(1, n - MS_LEN + 2):
        uids = [POS_UID[(pid, p)] for p in range(start, start + MS_LEN)]
        target_candidates = [u for u in uids if u in eligible_target_uids]
        key = (int(year), pid, int(start))
        hd_pop_rows.append({
            "year": int(year),
            "pid": pid,
            "start": int(start),
            "population_ms_key": f"{int(year)}:{pid}:{int(start)}",
            "in_original_set": key in original_window_keys,
            "n_target_candidates": len(target_candidates),
            "target_candidates": "|".join(target_candidates),
            "ms_len": MS_LEN,
            "exposure": MS_LEN / PID_N[pid],
        })
hd_ms_population = pd.DataFrame(hd_pop_rows)
hd_ms_candidates = hd_ms_population[
    (~hd_ms_population.in_original_set) & (hd_ms_population.n_target_candidates > 0)
].copy()

hd_ms_population_summary = pd.DataFrame([{
    "population": "all_len7_ms_from_original_patients",
    "n_ms": len(hd_ms_population),
    "patients": int(hd_ms_population.pid.nunique()),
    "target_eligible_ms_excluding_original_ms": len(hd_ms_candidates),
    "target_eligible_patients": int(hd_ms_candidates.pid.nunique()) if len(hd_ms_candidates) else 0,
    "rule": "all length-7 mini-sequences from original-set patients; original-set mini-sequences excluded; overlap targets excluded",
}])
print(f"hidden duplicate mini-sequence population: {len(hd_ms_population)}; target-eligible candidates: {len(hd_ms_candidates)}")
display(hd_ms_population_summary.T)

def target_stratum(uid, vec_now, n_now):
    deficits = (target_vec - vec_now / n_now) if n_now else target_vec.copy()
    cells = [ri * 5 + int(IMG.loc[uid, f"orig_{r}"]) for ri, r in enumerate(ROIS)]
    ci = max(cells, key=lambda c: deficits[c])
    return f"{ROIS[ci // 5]}x{ci % 5}"

first_ms_of = occ.sort_values("ms_key").groupby("uid").ms_key.first()
cur = L1_before
selected_idx = set()
used_targets = set()
rep_occ_rows, rep_info_rows, rep_target_rows = [], [], []
candidate_records = hd_ms_candidates.reset_index(drop=True).to_dict("records")
while True:
    best = None
    for j, row in enumerate(candidate_records):
        if j in selected_idx:
            continue
        targets = [u for u in row["target_candidates"].split("|") if u and u not in used_targets]
        if not targets:
            continue
        add_vec = sum(U_ALL[u] for u in targets)
        new_l1 = float(np.abs((dup_vec + add_vec) / (dup_n + len(targets)) - target_vec).sum())
        if best is None or new_l1 < best[0]:
            best = (new_l1, j, targets, row, add_vec)
    if best is None:
        break
    new_l1, j, targets, row, add_vec = best
    if not (cur - new_l1) >= 1.0 / (dup_n + len(targets)):
        break

    selected_idx.add(j)
    target_set = set(targets)
    rep_key = f"REP{len(rep_info_rows) + 1:03d}"
    pid = row["pid"]
    year = int(row["year"])
    start = int(row["start"])
    target_seq2 = {}
    target_strata = {uid: target_stratum(uid, dup_vec, dup_n) for uid in targets}
    for p in range(start, start + MS_LEN):
        uid = POS_UID[(pid, p)]
        is_target = uid in target_set
        if is_target:
            target_seq2[uid] = p - start + 1
        rep_occ_rows.append({
            "ms_key": rep_key,
            "year": year,
            "pid": pid,
            "uid": uid,
            "seq2": p - start + 1,
            "entry_codes": set(),
            "presentation_type": "hidden_duplicate",
            "dup_source": "stratified" if is_target else pd.NA,
            "dup_stratum": target_strata[uid] if is_target else pd.NA,
            "repeat_of": first_ms_of[uid] if is_target else pd.NA,
        })
    rep_info_rows.append({
        "ms_key": rep_key,
        "population_ms_key": row["population_ms_key"],
        "n_target_images": len(targets),
        "target_uids": "|".join(targets),
        "ms_len": MS_LEN,
        "year": year,
        "pid": pid,
        "start": start,
        "exposure": MS_LEN / PID_N[pid],
    })
    for uid in targets:
        rep_target_rows.append({
            "ms_key": rep_key,
            "population_ms_key": row["population_ms_key"],
            "target_uid": uid,
            "repeat_of": first_ms_of[uid],
            "dup_stratum": target_strata[uid],
            "target_seq2": target_seq2[uid],
            "year": year,
            "pid": pid,
            "start": start,
        })
    used_targets.update(targets)
    dup_vec += add_vec
    dup_n += len(targets)
    cur = new_l1

rep_occ_cols = ["ms_key", "year", "pid", "uid", "seq2", "entry_codes", "presentation_type", "dup_source", "dup_stratum", "repeat_of"]
rep_info_cols = ["ms_key", "population_ms_key", "n_target_images", "target_uids", "ms_len", "year", "pid", "start", "exposure"]
rep_target_cols = ["ms_key", "population_ms_key", "target_uid", "repeat_of", "dup_stratum", "target_seq2", "year", "pid", "start"]
rep_occ = pd.DataFrame(rep_occ_rows, columns=rep_occ_cols)
rep_info = pd.DataFrame(rep_info_rows, columns=rep_info_cols)
rep_targets = pd.DataFrame(rep_target_rows, columns=rep_target_cols)
rep_links = rep_targets[["ms_key", "repeat_of"]].drop_duplicates() if len(rep_targets) else pd.DataFrame(columns=["ms_key", "repeat_of"])
L1_after = float(np.abs(dup_vec / dup_n - target_vec).sum()) if dup_n else float("inf")

if len(rep_info):
    assert rep_info.population_ms_key.is_unique, "same hidden duplicate mini-sequence selected twice"
    assert rep_targets.target_uid.is_unique, "supplement target image selected twice"
    assert rep_occ.presentation_type.eq("hidden_duplicate").all()
    assert (rep_occ.groupby("ms_key").size() == MS_LEN).all()

hidden_targets = pd.concat([
    pd.DataFrame({"uid": list(overlap_uids.index), "dup_source": "overlap"}),
    rep_targets.rename(columns={"target_uid": "uid"})[["uid"]].assign(dup_source="stratified")
], ignore_index=True)
dup_grade_counts = (hidden_targets.merge(IMG[[f"orig_{r}" for r in ROIS]], left_on="uid", right_index=True, how="left")
                    .melt(id_vars=["uid", "dup_source"], value_vars=[f"orig_{r}" for r in ROIS], var_name="ROI", value_name="orig_grade"))
dup_grade_counts["ROI"] = dup_grade_counts.ROI.str.replace("orig_", "", regex=False)
stratum_alloc = (dup_grade_counts.groupby(["dup_source", "ROI", "orig_grade"])
                 .agg(images=("uid", "nunique")).reset_index())

duplicate_scale_decision = pd.DataFrame([{
    "overlap_secondpass_targets": overlap_n,
    "selected_ms_population_total": len(hd_ms_population),
    "selected_ms_population_target_eligible": len(hd_ms_candidates),
    "selected_ms_population_picked": len(rep_info),
    "supplement_target_images": len(rep_targets),
    "supplement_presented_rows": len(rep_occ),
    "total_hidden_duplicate_targets": dup_n,
    "L1_before_supplement": round(L1_before, 4),
    "L1_after_supplement": round(L1_after, 4),
    "stop_threshold_last": round(1.0 / dup_n, 6) if dup_n else pd.NA,
    "RB_grade3_targets": int(((dup_grade_counts.ROI == "RB") & (dup_grade_counts.orig_grade == 3)).sum()),
    "RB_grade4_targets": int(((dup_grade_counts.ROI == "RB") & (dup_grade_counts.orig_grade == 4)).sum()),
}])
print(f"hidden duplicate targets: overlap {overlap_n} + supplement {len(rep_targets)} = {dup_n}; supplement mini-sequences {len(rep_info)}; L1 {L1_before:.3f} -> {L1_after:.3f}")
display(duplicate_scale_decision.T)
display(stratum_alloc.pivot_table(index=["ROI", "orig_grade"], columns="dup_source", values="images", fill_value=0))


## 3.1, 3.4, 3.5, and 7 Presentation order and blinded reread files

Randomize mini-sequence order under gap constraints and assign blinded reread identifiers. Hidden-duplicate mini-sequences are placed after the original presentation of each target image they contain.


In [ ]:
MIN_PRESENTATION_GAP = 5         # 3.5 — 사전 지정
occ["presentation_type"] = pd.NA
occ["dup_source"] = pd.NA
occ["dup_stratum"] = pd.NA
occ["repeat_of"] = pd.NA
allocc = pd.concat([occ, rep_occ], ignore_index=True)

content = allocc.groupby("ms_key").uid.apply(frozenset).to_dict()
patient_of = allocc.groupby("ms_key").pid.first().to_dict()
keys = sorted(content)

pair_set = {}
def add_pair(a, b, typ):
    if a == b:
        return
    k = (min(a, b), max(a, b))
    pair_set.setdefault(k, typ)

for r in rep_links.to_dict("records"):             # original-hidden duplicate pairs
    add_pair(r["ms_key"], r["repeat_of"], "original-hidden duplicate")
uid_ms = {}
for k in keys:
    for u in content[k]:
        uid_ms.setdefault(u, []).append(k)
for u, ml in uid_ms.items():                         # ② 영상 공유(겹침)
    for i in range(len(ml)):
        for j in range(i + 1, len(ml)):
            add_pair(ml[i], ml[j], "겹침 공유")
pid_ms = {}
for k in keys:
    pid_ms.setdefault(patient_of[k], []).append(k)
for ml in pid_ms.values():                           # ③ 같은 환자
    for i in range(len(ml)):
        for j in range(i + 1, len(ml)):
            add_pair(ml[i], ml[j], "같은 환자")
pairs = list(pair_set)
print(f"mini-sequence {len(keys)}개 · 간격 제약 쌍 {len(pairs)}개")

pairs_of = {}
for a, b in pairs:
    pairs_of.setdefault(a, []).append((a, b))
    pairs_of.setdefault(b, []).append((a, b))

perm = list(rng_order.permutation(keys))
pos = {m: i for i, m in enumerate(perm)}
def all_viols():
    return [p for p in pairs if abs(pos[p[0]] - pos[p[1]]) < MIN_PRESENTATION_GAP]
def local_v(m):
    return sum(1 for a, b in pairs_of.get(m, ())
               if abs(pos[a] - pos[b]) < MIN_PRESENTATION_GAP)

viols = all_viols()
steps = 0
while viols and steps < 100000:
    steps += 1
    a, b = viols[int(rng_order.integers(len(viols)))]
    mover = a if rng_order.random() < 0.5 else b
    j = int(rng_order.integers(len(perm)))
    other = perm[j]
    if other == mover:
        continue
    before = local_v(mover) + local_v(other)
    pi_, pj = pos[mover], j
    perm[pi_], perm[pj] = perm[pj], perm[pi_]
    pos[mover], pos[other] = pj, pi_
    after = local_v(mover) + local_v(other)
    if after > before:
        perm[pi_], perm[pj] = perm[pj], perm[pi_]
        pos[mover], pos[other] = pi_, pj
    elif after < before:
        viols = all_viols()
if all_viols():
    raise ValueError("최소 제시 간격을 만족하는 배치를 찾지 못했다")
print(f"국소 교환 {steps}회로 간격 제약 충족")

for _ in range(1000):                                # hidden duplicate after original target
    bad = [r for r in rep_links.to_dict("records") if pos[r["ms_key"]] <= pos[r["repeat_of"]]]
    if not bad:
        break
    r = bad[0]
    i, j = pos[r["ms_key"]], pos[r["repeat_of"]]
    perm[i], perm[j] = perm[j], perm[i]
    pos[r["ms_key"]], pos[r["repeat_of"]] = j, i
    if all_viols():
        perm[i], perm[j] = perm[j], perm[i]
        pos[r["ms_key"]], pos[r["repeat_of"]] = i, j
        break
assert not all_viols()
assert all(pos[r["ms_key"]] > pos[r["repeat_of"]] for r in rep_links.to_dict("records"))

ms_rank = {m: i + 1 for i, m in enumerate(perm)}
allocc["ms_rank"] = allocc.ms_key.map(ms_rank)
schedule = allocc.sort_values(["ms_rank", "seq2"]).reset_index(drop=True)
schedule["read_order"] = np.arange(1, len(schedule) + 1)
schedule["ms_id"] = "P2A_M" + schedule.ms_rank.astype(str).str.zfill(3)
schedule["uid2"] = "P2A_I" + schedule.read_order.astype(str).str.zfill(5)

untyped = schedule.presentation_type.isna()
is_first = ~schedule.uid.duplicated(keep="first")
schedule.loc[untyped & is_first, "presentation_type"] = "original"
schedule.loc[untyped & ~is_first, "presentation_type"] = "hidden_duplicate"
schedule.loc[untyped & ~is_first, "dup_source"] = "overlap"
assert not (schedule.presentation_type.eq("original")
            & schedule.ms_key.str.startswith("REP")).any(), "반복 행이 최초 제시로 분류됨"
assert int(schedule.presentation_type.eq("original").sum()) == len(sel_uids)

print("제시 유형")
display(schedule.groupby(["presentation_type", "dup_source"], dropna=False)
        .agg(rows=("uid2", "size"), unique_images=("uid", "nunique")))
print(f"총 제시 {len(schedule)}장 · 고유 영상 {schedule.uid.nunique()}장 · "
      f"mini-sequence {len(keys)}개")

gap_rows = [{"pair_type": t, "ms_a": a, "ms_b": b,
             "gap": abs(ms_rank[a] - ms_rank[b])}
            for (a, b), t in pair_set.items()]
presentation_gaps = pd.DataFrame(gap_rows)
print("제시 간격 분포")
display(presentation_gaps.groupby("pair_type").gap.describe()[["count", "min", "50%", "max"]])

# ── 연도 간 제시 범위 대칭 (3.4) ──────────────────────────────────────────
orig_ms = ms_info.copy()
assert (orig_ms.ms_len == MS_LEN).all(), "길이 7 고정 위반"
a24 = orig_ms[orig_ms.year.eq(2024)]
a26 = orig_ms[orig_ms.year.eq(2026)]
exposure_symmetry = pd.DataFrame({
    "2024": [len(a24), MS_LEN, round(a24.exposure.median(), 3),
             round(float(mannwhitneyu(a24.exposure, a26.exposure).pvalue), 4)],
    "2026": [len(a26), MS_LEN, round(a26.exposure.median(), 3),
             round(float(ks_2samp(a24.exposure, a26.exposure).pvalue), 4)],
}, index=["mini_sequence", "length_fixed", "exposure_median", "MW_p/KS_p"])
display(exposure_symmetry)

ms_per_patient = (orig_ms.groupby(["year", "pid"]).size().reset_index(name="n_ms"))
print("환자당 mini-sequence 수 분포")
display(ms_per_patient.pivot_table(index="n_ms", columns="year",
                                   values="pid", aggfunc="size", fill_value=0))

# ── 재판독 축 산출물 ───────────────────────────────────────────
reread_manifest = schedule[["read_order", "ms_id", "seq2", "uid2"]].copy()
reread_template = reread_manifest.copy()
for c in ["new_RT", "new_LT", "new_RB", "new_LB", "evaluable", "non_evaluable_reason"]:
    reread_template[c] = pd.NA

forbidden = {"pid", "uid", "seq", "year", "orig_RT", "orig_LT", "orig_RB", "orig_LB",
             "entry_path", "presentation_type", "dup_source", "dup_stratum",
             "pi_ms", "pi_uid", "weight_stratum", "repeat_of", "ms_key",
             "denom_role_RT", "denom_role_LT", "denom_role_RB", "denom_role_LB"}
assert forbidden.isdisjoint(reread_manifest.columns)
assert forbidden.isdisjoint(reread_template.columns)
print(f"판독 목록 {len(reread_manifest)}장 · 반환 양식 {len(reread_template)}장 — "
      "비노출 항목(기존 라벨·연도·선정 경로·분모 자격·중복 여부·환자 식별자) 부재 확인 (3.1)")
display(reread_manifest.head(8))


## 8. Save descriptive outputs and single 1,155-row master file

Write one master CSV for every presented row. This file is the source for NPZ construction and downstream analysis checks.


In [ ]:
OUTPUT_DIR = Path("Result/Phase2A_v2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def save_csv(table, rel_path):
    path = OUTPUT_DIR / rel_path
    path.parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(path, index=False, encoding="utf-8-sig")
    return path

# ---- descriptive statistics first -------------------------------------------------
cohort_summary = (
    df.groupby("year")
      .agg(images=("uid", "size"), patients=("pid", "nunique"))
      .reset_index()
)

patient_image_counts = (
    df.groupby(["year", "pid"])
      .agg(images=("uid", "size"))
      .reset_index()
)

patient_image_count_summary = (
    patient_image_counts.groupby("year")["images"]
    .describe(percentiles=[0.25, 0.5, 0.75])
    .reset_index()
)

roi_grade_distribution = (
    df.melt(
        id_vars=["year", "pid", "uid"],
        value_vars=[f"orig_{roi}" for roi in ROIS],
        var_name="ROI",
        value_name="old_grade",
    )
)
roi_grade_distribution["ROI"] = roi_grade_distribution["ROI"].str.replace("orig_", "", regex=False)
roi_grade_distribution = (
    roi_grade_distribution
    .groupby(["year", "ROI", "old_grade"])
    .agg(images=("uid", "nunique"), patients=("pid", "nunique"))
    .reset_index()
    .sort_values(["year", "ROI", "old_grade"])
)

rb_grade_distribution = (
    df.groupby(["year", "orig_RB"])
      .agg(images=("uid", "size"), patients=("pid", "nunique"))
      .reset_index()
      .rename(columns={"orig_RB": "old_RB"})
)

eligibility_summary = group_eligibility.copy()
eligibility_overlap = patient_overlap.copy()

# ---- row-level master -------------------------------------------------------------
presented = df[df.uid.isin(schedule.uid.unique())][
    ["uid", "pid", "seq", "seq_position", "year", "image_path"] + [f"orig_{roi}" for roi in ROIS]
].copy()

for roi in ROIS:
    presented[f"role_{roi}"] = presented[f"orig_{roi}"].map(
        lambda g: "|".join(DERIVE.get(int(g), []))
    )

def flatten_selection_role(values):
    roles = set()
    for val in values:
        if pd.isna(val):
            continue
        if isinstance(val, (set, list, tuple)):
            parts = val
        else:
            parts = str(val).replace("{", "").replace("}", "").replace("'", "").split("|")
        for part in parts:
            part = str(part).strip()
            if part:
                roles.add(part)
    return "|".join(sorted(roles))

selection_by_uid = occ.groupby("uid").entry_codes.apply(flatten_selection_role).to_dict()

master = schedule[["read_order", "ms_id", "seq2", "uid2", "uid"]].copy()
master["read_type"] = np.where(master.uid.duplicated(keep="first"), "repeat_read", "first_read")
master["selection_role"] = master.uid.map(selection_by_uid)
master["selection_role"] = master["selection_role"].fillna("repeat_minisequence")

master = master.merge(
    presented[
        ["uid", "pid", "seq", "year", "image_path",
         "orig_RT", "orig_LT", "orig_RB", "orig_LB",
         "role_RT", "role_LT", "role_RB", "role_LB"]
    ],
    on="uid",
    how="left",
)

master = master.rename(columns={
    "orig_RT": "old_RT",
    "orig_LT": "old_LT",
    "orig_RB": "old_RB",
    "orig_LB": "old_LB",
})

master = master[
    ["read_order", "ms_id", "seq2", "uid2", "uid", "pid", "year", "seq", "image_path",
     "read_type", "selection_role",
     "old_RT", "old_LT", "old_RB", "old_LB",
     "role_RT", "role_LT", "role_RB", "role_LB"]
].sort_values("read_order").reset_index(drop=True)

# ---- validation gates -------------------------------------------------------------
assert len(master) == 1155, len(master)
assert master.ms_id.nunique() == 165, master.ms_id.nunique()
assert master.groupby("ms_id").size().eq(7).all()
assert master.uid2.is_unique
assert master.selection_role.notna().all()
assert master.selection_role.astype(str).str.len().gt(0).all()
assert not master.selection_role.astype(str).str.contains(r"[{}']", regex=True).any()
for col in ["role_RT", "role_LT", "role_RB", "role_LB"]:
    assert master[col].notna().all(), col
assert set(master.read_type) == {"first_read", "repeat_read"}

lock_summary = pd.Series({
    "ms_len": MS_LEN,
    "mini_sequence_count": int(master.ms_id.nunique()),
    "total_presented_rows": len(master),
    "presented_unique_images": int(master.uid.nunique()),
    "first_read_rows": int(master.read_type.eq("first_read").sum()),
    "repeat_read_rows": int(master.read_type.eq("repeat_read").sum()),
    "source_csv": str(SOURCE_CSV),
    "master_csv": "phase2A_master_1155.csv",
}, name="value").to_frame().reset_index()

outputs = {
    "01_descriptive/cohort_summary.csv": cohort_summary,
    "01_descriptive/patient_image_counts.csv": patient_image_counts,
    "01_descriptive/patient_image_count_summary.csv": patient_image_count_summary,
    "01_descriptive/roi_grade_distribution.csv": roi_grade_distribution,
    "01_descriptive/rb_grade_distribution.csv": rb_grade_distribution,
    "01_descriptive/eligibility_summary.csv": eligibility_summary,
    "01_descriptive/eligibility_overlap.csv": eligibility_overlap,
    "phase2A_master_1155.csv": master,
    "phase2A_lock_summary.csv": lock_summary,
}

for rel_path, table in outputs.items():
    save_csv(table, rel_path)

print(f"saved: {OUTPUT_DIR.resolve()}")
display(pd.DataFrame([{"file": k, "rows": len(v)} for k, v in outputs.items()]))
display(lock_summary)
display(master.head(10))
